In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets, metrics, model_selection
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
import statsmodels.formula.api as smf


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
file_path = '/kaggle/input/datasets/rutuspatel/walmart-dataset-retail/Walmart_Store_sales.csv'

df = pd.read_csv(file_path)
df

In [ ]:
df.head()


In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)



In [ ]:
df.dtypes


In [ ]:
weekly_total = df.groupby('Date')['Weekly_Sales'].sum()

plt.figure(figsize=(20,5))
plt.plot(weekly_total)
plt.title('Total Weekly Retail Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Weekly Sales')
plt.show()


In [ ]:
df['Store'] = df['Store'].astype('category')

In [ ]:
df.duplicated(subset=['Store', 'Date']).sum()


In [ ]:
df['Store'].nunique()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

stores_to_plot = df['Store'].cat.categories[:5]
plt.figure(figsize=(10,5))

for s in stores_to_plot:
    tmp = df[df['Store'] == s].sort_values('Date')
    plt.plot(tmp['Date'], tmp['Weekly_Sales'], label=f"Store {int(s)}")

plt.title("Weekly Sales Over Time (Sample Stores)")
plt.xlabel("Date")
plt.ylabel("Weekly Sales")

# ✅ Monthly ticks
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())                 # every month
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))       # format label

plt.ticklabel_format(style='plain', axis='y')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
store_totals = df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,5))
store_totals.head(20).plot(kind='bar')
plt.title("Top 20 Stores by Total Sales")
plt.xlabel("Store")
plt.ylabel("Total Sales")
plt.ticklabel_format(style='plain', axis='y') 
plt.show()


In [ ]:
df['Date'].min(), df['Date'].max()


In [ ]:
df = df.sort_values('Date')
cut1 = df['Date'].quantile(0.70)
cut2 = df['Date'].quantile(0.85)

train = df[df['Date'] <= cut1].copy()
val   = df[(df['Date'] > cut1) & (df['Date'] <= cut2)].copy()
test  = df[df['Date'] > cut2].copy()


In [ ]:
df = df.sort_values(['Store', 'Date'])
df['lag_1'] = df.groupby('Store', observed = True)['Weekly_Sales'].shift(1)


In [ ]:
df['lag_1'].isna().sum()


In [ ]:
df['Date'].min(), df['Date'].max(), df['Store'].nunique()


In [ ]:
df

In [ ]:
df = df.sort_values(['Store', 'Date'])

for lag in [1, 2, 4, 12, 52]:
    df[f'lag_{lag}'] = df.groupby('Store', observed = True)['Weekly_Sales'].shift(lag)


In [ ]:
df['roll_mean_4']  = df.groupby('Store', observed = True)['Weekly_Sales'].shift(1).rolling(4).mean()
df['roll_mean_12'] = df.groupby('Store', observed = True)['Weekly_Sales'].shift(1).rolling(12).mean()


In [ ]:
df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
df['month']        = df['Date'].dt.month
df['year']         = df['Date'].dt.year


In [ ]:
df['pre_holiday']  = df.groupby('Store', observed = True)['Holiday_Flag'].shift(-1)
df['post_holiday'] = df.groupby('Store', observed = True)['Holiday_Flag'].shift(1)


In [ ]:
features = [
    'lag_1','lag_2','lag_4','lag_12','lag_52',
    'roll_mean_4','roll_mean_12',
    'Holiday_Flag','pre_holiday','post_holiday',
    'Temperature','Fuel_Price','CPI','Unemployment',
    'week_of_year','month','year','Store'
]

df_model = df.dropna(subset=features + ['Weekly_Sales'])

train = df_model[df_model['Date'] <= '2011-12-30']
val   = df_model[(df_model['Date'] > '2011-12-30') & (df_model['Date'] <= '2012-06-29')]
test  = df_model[df_model['Date'] > '2012-06-29']


In [ ]:
X_train = pd.get_dummies(train[features], drop_first=True)
X_val   = pd.get_dummies(val[features], drop_first=True)
X_test  = pd.get_dummies(test[features], drop_first=True)

X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train['Weekly_Sales']
y_val   = val['Weekly_Sales']
y_test  = test['Weekly_Sales']


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
mae = mean_absolute_error(y_val, val_preds)
mae


In [ ]:
val_baseline = val.dropna(subset=['lag_1'])
baseline_mae = mean_absolute_error(
    val_baseline['Weekly_Sales'], 
    val_baseline['lag_1']
)

baseline_mae


In [ ]:
baseline_mae, mae


In [ ]:
test_preds = model.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae_pct = test_mae / y_test.mean() * 100
test_rmse_pct = test_rmse /  y_test.mean() * 100

print(f"MAE Percent Error: {test_mae_pct}")

print(f"RMSE Percent Error: {test_rmse_pct}")


In [ ]:
bmodel_mae = mean_absolute_error(y_test, test_preds)

print(bmodel_mae)

In [ ]:
nmae_pct = bmodel_mae / y_test.mean() * 100
print(nmae_pct)

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_val, val_preds))
rmse


In [ ]:
avg_sales = y_val.mean()
mae_pct = mae / avg_sales * 100
mae_pct


In [ ]:
intercept = model.intercept_
print("Intercept:", intercept)

for feature, coef in coef_series.items():
    print(f"{feature}: {coef}")


In [ ]:
coef_series.sort_values(key=abs, ascending=False).head(10)


In [ ]:
df


In [ ]:
file_path = '/kaggle/input/datasets/rutuspatel/walmart-dataset-retail/Walmart_Store_sales.csv'

yes = pd.read_csv(file_path)
yes

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


df['lag1_baseline'] = df['Weekly_Sales'].shift(1)


df_baseline = df.dropna(subset=['lag1_baseline'])

y_actual = df_baseline['Weekly_Sales']
baseline_prediction = df_baseline['lag1_baseline']

baseline_rmse = np.sqrt(mean_squared_error(y_actual, baseline_prediction))
baseline_mape = mean_absolute_percentage_error(y_actual, baseline_prediction) * 100


mean_sales = y_actual.mean()
mae_percent = (baseline_mae / mean_sales) * 100
rmse_percent = (baseline_rmse / mean_sales) * 100

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline MAE %:", mae_percent, "%")
print("Baseline RMSE %:", rmse_percent, "%")

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(y_actual.values[:200], label="Actual Weekly Sales")
plt.plot(baseline_prediction.values[:200], label="Lag-1 Baseline Prediction", linestyle="--")

plt.title("Lag-1 Baseline Forecast vs Actual Sales (First 200 Weeks)")
plt.xlabel("Time (Weeks)")
plt.ylabel("Weekly Sales")
plt.legend()

plt.show()

In [ ]:
import pandas as pd

coef_series = pd.Series(model.coef_, index=X_train.columns)
print(coef_series)

In [ ]:
print(list(df.columns))
print(df.info())
print(df.describe())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

coef_df = coef_series.sort_values()

plt.figure(figsize=(10, 8))
coef_df.plot(kind='barh')
plt.title("Sorted Feature Coefficients")
plt.xlabel("Coefficient Value")
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, test_preds, alpha=0.3)

plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red')

plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales')

plt.ticklabel_format(style='plain', axis='both')  

plt.show()


In [ ]:
plt.plot(y_test.values, label="Actual")
plt.plot(test_preds, label="Predicted")

plt.xlabel("Test Sample Index")
plt.ylabel("Sales")

plt.ticklabel_format(style='plain', axis='y')

plt.legend()
plt.show()


In [ ]:
plt.plot(y_test.index, y_test.values, label="Actual")
plt.plot(y_test.index, test_preds, label="Predicted")

plt.xlabel("Time")
plt.ylabel("Sales")

plt.ticklabel_format(style='plain', axis='y')

plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Convert to dictionary if not already
coef_dict = coef_series.to_dict()

# FILTER: keep only lag variables
lag_coef_dict = {k: v for k, v in coef_dict.items() if "lag_" in k.lower()}

features = list(lag_coef_dict.keys())
values = list(lag_coef_dict.values())

plt.figure(figsize=(10, 8))
plt.barh(features, values)
plt.xlabel("Coefficient Value")
plt.title("Lag Feature Coefficients")
plt.axvline(0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates

plt.plot(y_test.index, y_test.values, label="Actual")
plt.plot(y_test.index, test_preds, label="Predicted")

plt.xlabel("Time")
plt.ylabel("Sales")

# Format dates nicely
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

plt.xticks(rotation=45)

plt.ticklabel_format(style='plain', axis='y')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(type(y_test.index))


In [ ]:

df['Date'] = pd.to_datetime(df['Date'])


X = df.set_index('Date')

In [ ]:
selected_stores = df['Store'].unique()[:5]  

In [ ]:
filtered_idx = df.loc[y_test.index]['Store'].isin(selected_stores)

y_test_filtered = y_test[filtered_idx]
test_preds_filtered = test_preds[filtered_idx]
test_dates_filtered = df.loc[y_test_filtered.index, 'Date']
store_filtered = df.loc[y_test_filtered.index, 'Store']

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10,15), sharex=True)

for i, store in enumerate(selected_stores):
    store_mask = store_filtered == store
    
    axes[i].plot(
        test_dates_filtered[store_mask],
        y_test_filtered[store_mask],
        label="Actual"
    )
    
    axes[i].plot(
        test_dates_filtered[store_mask],
        test_preds_filtered[store_mask],
        linestyle='--',
        label="Predicted"
    )
    
    axes[i].set_title(f"Store {store}")
    axes[i].ticklabel_format(style='plain', axis='y')
    axes[i].legend()
padding = (y_max - y_min) * .5
plt.ylim(y_min - padding, y_max + padding)
plt.suptitle("Actual vs Predicted Weekly Sales Across Selected Stores", fontsize=16)
plt.xlabel("Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

for store in selected_stores:
    store_mask = store_filtered == store
    
    plt.plot(
        test_dates_filtered[store_mask],
        y_test_filtered[store_mask],
        label=f"Actual Store {store}"
    )
    
    plt.plot(
        test_dates_filtered[store_mask],
        test_preds_filtered[store_mask],
        linestyle='--',
        label=f"Pred Store {store}"
    )

plt.xlabel("Time")
plt.ylabel("Sales")


y_min = min(y_test_filtered.min(), test_preds_filtered.min())
y_max = max(y_test_filtered.max(), test_preds_filtered.max())
padding = (y_max - y_min) * 0.1
plt.ylim(y_min - padding, y_max + padding)

plt.ticklabel_format(style='plain', axis='y')
plt.xticks(rotation=45)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

plt.figure(figsize=(8,6))
plt.scatter(test_preds, residuals, alpha=0.3)

z = np.polyfit(test_preds, residuals, 1)
p = np.poly1d(z)
plt.plot(test_preds, p(test_preds))

plt.axhline(0)

plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot with Trend Line")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

plt.figure(figsize=(6,6))

ax = plt.gca()

df.boxplot(column="Weekly_Sales", by="Holiday_Flag", ax=ax)


ax.yaxis.set_major_formatter(ScalarFormatter())
ax.ticklabel_format(style='plain', axis='y')

plt.xticks([1, 2], ["Pre_holiday", "Holiday"])
plt.xlabel("Week Type")
plt.ylabel("Sales")
plt.title("Sales Distribution: Holiday vs Pre-Holiday")
plt.suptitle("")

plt.tight_layout()
plt.show()